# Predictions using trained model

In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [2]:
## Load the trained model, scalers and encoders
model = load_model("best_model.keras")

with open("label_encoder_gender.pkl", 'rb') as file:
    gender_encoder = pickle.load(file)

with open("one_hot_encoder_geography.pkl", "rb") as file:
    geography_encoder = pickle.load(file)

with open("sclaer.pkl", "rb") as file:
    scaler = pickle.load(file)

/Users/vedanshdwivedi/ai_course/venv/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 18 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(store)


In [3]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [15]:
def get_transformed_data(input: dict) -> dict:
    data = input.copy()
    encoded_gender, [geo_fr, geo_gr, geo_sp] = gender_encoder.transform([data["Gender"]])[0], geography_encoder.transform([[data["Geography"]]])[0]
    data["Gender"] = int(encoded_gender)
    data["Geography_France"] = int(geo_fr)
    data["Geography_Spain"] = int(geo_sp)
    data["Geography_Germany"] = int(geo_gr)
    del data["Geography"]
    return data

In [16]:
def get_scaled_data(input: dict):
    df = pd.DataFrame([input])
    df = df[scaler.feature_names_in_]
    scaled_array = scaler.transform(df)
    return scaled_array

In [19]:
def get_prediction(data):
    input = data.copy()
    new_data = get_transformed_data(input)
    new_data_scaled = get_scaled_data(new_data)
    prediction = model.predict(new_data_scaled)
    churn_probability = float(prediction[0][0])
    print(f"Churn Probability: {churn_probability:.2%}")
    if churn_probability > 0.5:
        print("Prediction: Customer will CHURN (Exit)")
    else:
        print("Prediction: Customer will STAY")

In [20]:
get_prediction(input_data)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
Churn Probability: 3.33%
Prediction: Customer will STAY


/Users/vedanshdwivedi/ai_course/venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
